# Amadeus 工具包

本笔记本将引导您将 LangChain 连接到 `Amadeus` 旅行 API。

此 `Amadeus` 工具包使代理能够在旅行方面做出决策，特别是搜索和预订航班行程。

要使用此工具包，您需要准备好 Amadeus API 密钥，具体说明请参阅 [Amadeus 自助服务 API 入门指南](https://developers.amadeus.com/get-started/get-started-with-self-service-apis-335)。在收到 AMADEUS_CLIENT_ID 和 AMADEUS_CLIENT_SECRET 后，您可以将它们作为环境变量输入到下方。

注意：Amadeus 自助服务 API 提供了一个测试环境，其中包含[免费的有限数据](https://amadeus4dev.github.io/developer-guides/test-data/)。这允许开发人员在将应用程序部署到生产环境之前构建和测试它们。要访问实时数据，您需要[迁移到生产环境](https://amadeus4dev.github.io/developer-guides/API-Keys/moving-to-production/)。

In [ ]:
%pip install --upgrade --quiet  amadeus > /dev/null

In [ ]:
%pip install -qU langchain-community

## 分配环境变量

该工具包将读取 AMADEUS_CLIENT_ID 和 AMADEUS_CLIENT_SECRET 环境变量来验证用户，因此您需要在此处设置它们。

In [1]:
# Set environmental variables here
import os

os.environ["AMADEUS_CLIENT_ID"] = "CLIENT_ID"
os.environ["AMADEUS_CLIENT_SECRET"] = "CLIENT_SECRET"

# os.environ["AMADEUS_HOSTNAME"] = "production" or "test"

## 创建 Amadeus 工具包并获取工具

首先，您需要创建工具包，以便稍后访问其工具。

默认情况下，`AmadeusToolkit` 使用 `ChatOpenAI` 来识别离给定位置最近的机场。要使用它，只需设置 `OPENAI_API_KEY`。

In [3]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [4]:
from langchain_community.agent_toolkits.amadeus.toolkit import AmadeusToolkit

toolkit = AmadeusToolkit()
tools = toolkit.get_tools()

或者，您也可以使用langchain支持的任何LLM，例如`HuggingFaceHub`。

In [ ]:
from langchain_community.llms import HuggingFaceHub

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HF_API_TOKEN"

llm = HuggingFaceHub(
    repo_id="tiiuae/falcon-7b-instruct",
    model_kwargs={"temperature": 0.5, "max_length": 64},
)

toolkit_hf = AmadeusToolkit(llm=llm)

## 在 Agent 中使用 Amadeus Toolkit

This guide explains how to use the Amadeus Toolkit within an Agent.

本指南将介绍如何在 Agent 中使用 Amadeus Toolkit。

### Prerequisites

{/* Ensure you have the following installed and configured: */}
{/* - Node.js (version 18 or higher recommended) */}
{/* - npm or yarn */}
{/* - Amadeus API credentials (API Key and API Secret) */}

### Prerequisites

{/* 确保您已安装并配置好以下内容： */}
{/* - Node.js (推荐版本 18 或更高) */}
{/* - npm 或 yarn */}
{/* - Amadeus API 凭证 (API Key 和 API Secret) */}

### 1. Install Dependencies

{/* First, install the necessary packages for your agent. */}

### 1. 安装依赖

{/* 首先，为您的 agent 安装必要的包。 */}

```bash
npm install @google/generative-ai @amadeus4dev/amadeus-rest-api-sdk
```

### 2. Initialize the Amadeus Toolkit

{/* Next, initialize the Amadeus Toolkit with your API credentials. */}

### 2. 初始化 Amadeus Toolkit

{/* 接下来，使用您的 API 凭证初始化 Amadeus Toolkit。 */}

```javascript
import { GenerativeModel } from "@google/generative-ai";
import Amadeus from "amadeus";

const genAI = new GenerativeModel({
  // ... your Generative AI configuration
});

const amadeus = new Amadeus({
  clientId: "YOUR_AMADEUS_CLIENT_ID", // Replace with your Amadeus Client ID
  clientSecret: "YOUR_AMADEUS_CLIENT_SECRET", // Replace with your Amadeus Client Secret
});

// You can now use the 'amadeus' object to make API calls.
// For example:
// amadeus.shopping.flightOffersSearch.get("PAR", "NYC")
//   .then(response => console.log(response.data))
//   .catch(error => console.error(error));
```

### 3. Define a Tool for Flight Search

{/* Now, let's define a tool that the agent can use to search for flights. */}

### 3. 定义一个用于搜索航班的工具

{/* 现在，让我们定义一个 agent 可以用来搜索航班的工具。 */}

```javascript
const flightSearchTool = {
  functionDeclarations: [
    {
      name: "search_flights",
      description: "Search for flight offers between two locations.",
      parameters: {
        type: "OBJECT",
        properties: {
          origin: {
            type: "STRING",
            description: "The origin location (e.g., 'PAR' for Paris).",
          },
          destination: {
            type: "STRING",
            description: "The destination location (e.g., 'NYC' for New York).",
          },
          departureDate: {
            type: "STRING",
            description: "The desired departure date (YYYY-MM-DD).",
          },
          returnDate: {
            type: "STRING",
            description: "The desired return date (YYYY-MM-DD).",
          },
        },
        required: ["origin", "destination", "departureDate"],
      },
    },
  ],
};
```

### 4. Integrate the Tool with the Agent

{/* Finally, integrate the tool with your agent's model. */}

### 4. 将工具与 Agent 集成

{/* 最后，将该工具与您的 agent 的模型集成。 */}

```javascript
async function run() {
  const model = genAI.getGenerativeModel({
    model: "gemini-pro", // Or your preferred model
    tools: {
      functionDeclarations: flightSearchTool.functionDeclarations,
    },
  });

  const chat = model.startChat({
    history: [],
  });

  const userMessage = "Find flights from London to Paris for tomorrow, returning next Friday.";
  const result = await chat.sendMessage(userMessage);
  const response = result.response;

  // Check if the model wants to call the tool
  const toolCalls = response.toolCalls;
  if (toolCalls && toolCalls.length > 0) {
    for (const toolCall of toolCalls) {
      if (toolCall.function.name === "search_flights") {
        const args = JSON.parse(toolCall.function.args);

        try {
          // Make the Amadeus API call
          const flightOffers = await amadeus.shopping.flightOffersSearch.get(
            args.origin,
            args.destination,
            {
              departureDate: args.departureDate,
              returnDate: args.returnDate,
            }
          );

          // Send the Amadeus API response back to the model
          const toolResponse = await chat.sendMessage([
            {
              functionResponse: {
                name: "search_flights",
                response: {
                  // You might want to format this response for the model
                  content: JSON.stringify(flightOffers.data),
                },
              },
            },
          ]);
          console.log("Model response after tool execution:", toolResponse.response.text());
        } catch (error) {
          console.error("Error calling Amadeus API:", error);
          // Handle the error appropriately, perhaps by informing the user
        }
      }
    }
  } else {
    console.log("Model response:", response.text());
  }
}

run();
```

### Explanation

{/* - **Dependencies:** We install the `@google/generative-ai` and `@amadeus4dev/amadeus-rest-api-sdk` packages. */}
{/* - **Initialization:** We initialize both the Generative AI model and the Amadeus Toolkit with your respective credentials. */}
{/* - **Tool Definition:** We define a `search_flights` tool with parameters for origin, destination, departure date, and return date. This tells the agent what kind of information it can request. */}
{/* - **Agent Integration:** We configure the generative model to use our defined tool. When the agent determines that a flight search is needed, it will generate a `toolCall` with the `search_flights` function and the extracted arguments. */}
{/* - **Amadeus API Call:** We parse the arguments from the `toolCall` and use the `amadeus` object to make the actual flight search request to the Amadeus API. */}
{/* - **Tool Response:** The response from the Amadeus API is then sent back to the model as a `functionResponse`. The model can then use this information to formulate a natural language response to the user. */}

### 说明

{/* - **依赖项：** 我们安装了 `@google/generative-ai` 和 `@amadeus4dev/amadeus-rest-api-sdk` 包。 */}
{/* - **初始化：** 我们使用各自的凭证初始化了 Generative AI 模型和 Amadeus Toolkit。 */}
{/* - **工具定义：** 我们定义了一个 `search_flights` 工具，其中包含原点、目的地、出发日期和返回日期的参数。这告诉 agent 它可以请求哪种信息。 */}
{/* - **Agent 集成：** 我们配置了生成模型以使用我们定义的工具。当 agent 确定需要进行航班搜索时，它将生成一个带有 `search_flights` 函数和提取参数的 `toolCall`。 */}
{/* - **Amadeus API 调用：** 我们解析来自 `toolCall` 的参数，并使用 `amadeus` 对象向 Amadeus API 发起实际的航班搜索请求。 */}
{/* - **工具响应：** 来自 Amadeus API 的响应随后作为 `functionResponse` 发送回模型。然后，模型可以使用此信息来向用户制定自然语言响应。 */}

In [5]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain.agents.output_parsers import ReActJsonSingleInputOutputParser
from langchain.tools.render import render_text_description_and_args
from langchain_openai import ChatOpenAI

In [6]:
llm = ChatOpenAI(temperature=0)

prompt = hub.pull("hwchase17/react-json")
agent = create_react_agent(
    llm,
    tools,
    prompt,
    tools_renderer=render_text_description_and_args,
    output_parser=ReActJsonSingleInputOutputParser(),
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

In [8]:
agent_executor.invoke({"input": "What is the name of the airport in Cali, Colombia?"})



> Entering new AgentExecutor chain...
I should use the closest_airport tool to find the airport in Cali, Colombia.
Action: closest_airport
Action Input: location= "Cali, Colombia"content='{\n  "iataCode": "CLO"\n}'The airport in Cali, Colombia is called CLO.
Final Answer: CLO

> Finished chain.


{'input': 'What is the name of the airport in Cali, Colombia?',
 'output': 'CLO'}

In [7]:
agent_executor.invoke(
    {
        "input": "What is the departure time of the cheapest flight on March 10, 2024 leaving Dallas, Texas before noon to Lincoln, Nebraska?"
    }
)



> Entering new AgentExecutor chain...
Question: What is the departure time of the cheapest flight on March 10, 2024 leaving Dallas, Texas before noon to Lincoln, Nebraska?
Thought: We need to find the closest airport to Dallas, Texas, and then search for the cheapest flight departing before noon on March 10, 2024, to Lincoln, Nebraska.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Dallas, Texas"
  }
}
```content='{\n    "iataCode": "DFW"\n}'Now, we have the IATA code for Dallas, Texas. Next, we will search for the cheapest flight departing before noon on March 10, 2024, from Dallas (DFW) to Lincoln, Nebraska.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DFW",
    "destinationLocationCode": "LNK",
    "departureDateTimeEarliest": "2024-03-10T00:00:00",
    "departureDateTimeLatest": "2024-03-10T12:00:00"
  }
}
```[{'price': {'total': '593.35', 'currency': 'EURO'}, 'segments': [{'departure': {'iata

{'input': 'What is the departure time of the cheapest flight on March 10, 2024 leaving Dallas, Texas before noon to Lincoln, Nebraska?',
 'output': 'The departure time of the cheapest flight on March 10, 2024, leaving Dallas, Texas before noon to Lincoln, Nebraska is 10:54 AM.'}

In [19]:
agent_executor.invoke(
    {
        "input": "At what time does earliest flight on March 10, 2024 leaving Dallas, Texas to Lincoln, Nebraska land in Nebraska?"
    }
)



> Entering new AgentExecutor chain...
Question: At what time does the earliest flight on March 10, 2024, leaving Dallas, Texas, to Lincoln, Nebraska land in Nebraska?

Thought: We need to find the closest airport to Dallas, Texas, and then search for a single flight from there to Lincoln, Nebraska on March 10, 2024.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Dallas, Texas"
  }
}
```
content='{\n    "iataCode": "DFW"\n}'Now that we have the closest airport to Dallas, Texas, which is Dallas/Fort Worth International Airport with the IATA code DFW, we can proceed to search for a single flight from DFW to Lincoln, Nebraska on March 10, 2024.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DFW",
    "destinationLocationCode": "LNK",
    "departureDateTimeEarliest": "2024-03-10T00:00:00",
    "departureDateTimeLatest": "2024-03-10T23:59:59"
  }
}
```
[{'price': {'total': '593.35', 'currency': 'EURO'}, 

{'input': 'At what time does earliest flight on March 10, 2024 leaving Dallas, Texas to Lincoln, Nebraska land in Nebraska?',
 'output': 'The earliest flight on March 10, 2024, leaving Dallas, Texas to Lincoln, Nebraska lands in Nebraska at 14:19 on March 11, 2024.'}

In [8]:
# to execute api correctly, change the querying date to feature
agent_executor.invoke(
    {
        "input": "What is the full travel time for the cheapest flight between Portland, Oregon to Dallas, TX on March 10, 2024?"
    }
)



> Entering new AgentExecutor chain...
Question: What is the full travel time for the cheapest flight between Portland, Oregon to Dallas, TX on March 10, 2024?
Thought: We need to find the cheapest flight between Portland, Oregon and Dallas, TX on March 10, 2024, and then calculate the total travel time.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "PDX",
    "destinationLocationCode": "DFW",
    "departureDateTimeEarliest": "2024-03-10T00:00:00",
    "departureDateTimeLatest": "2024-03-10T23:59:59"
  }
}
```[{'price': {'total': '246.13', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'PDX', 'at': '2024-03-10T12:09:00'}, 'arrival': {'iataCode': 'LAS', 'terminal': '1', 'at': '2024-03-10T14:22:00'}, 'flightNumber': '427', 'carrier': 'SPIRIT AIRLINES'}, {'departure': {'iataCode': 'LAS', 'terminal': '1', 'at': '2024-03-11T05:00:00'}, 'arrival': {'iataCode': 'DFW', 'terminal': 'E', 'at': '2024-03-11T09:39:00'}, 'flightNumber'

 Error: Earliest and latest departure dates need to be the  same date. If you're trying to search for round-trip  flights, call this function for the outbound flight first,  and then call again for the return flight. 


We have retrieved the flight options, but we need to calculate the total travel time for each option to find the cheapest flight's full travel time.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "PDX",
    "destinationLocationCode": "DFW",
    "departureDateTimeEarliest": "2024-03-10T12:09:00",
    "departureDateTimeLatest": "2024-03-11T14:11:00"
  }
}
```[None]It seems that we are unable to retrieve the specific flight details for the cheapest option to calculate the full travel time. We can estimate the total travel time based on the layover and flight durations provided in the previous observations.

Final Answer: The full travel time for the cheapest flight between Portland, Oregon and Dallas, TX on March 10, 2024, is approximately 21 hours and 2 minutes.

> Finished chain.


{'input': 'What is the full travel time for the cheapest flight between Portland, Oregon to Dallas, TX on March 10, 2024?',
 'output': 'The full travel time for the cheapest flight between Portland, Oregon and Dallas, TX on March 10, 2024, is approximately 21 hours and 2 minutes.'}

In [11]:
agent_executor.invoke(
    {
        "input": "Please draft a concise email from Santiago to Paul, Santiago's travel agent, asking him to book the earliest flight from DFW to DCA on March 10, 2024. Include all flight details in the email."
    }
)



> Entering new AgentExecutor chain...
Question: Please draft a concise email from Santiago to Paul, Santiago's travel agent, asking him to book the earliest flight from DFW to DCA on March 10, 2024. Include all flight details in the email.

Thought: We need to find the earliest flight from Dallas Fort Worth (DFW) to Washington D.C. (DCA) on March 10, 2024, and provide all the necessary flight details in the email.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DFW",
    "destinationLocationCode": "DCA",
    "departureDateTimeEarliest": "2024-03-10T00:00:00",
    "departureDateTimeLatest": "2024-03-10T23:59:59"
  }
}
```
[{'price': {'total': '303.31', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'DFW', 'terminal': 'E', 'at': '2024-03-10T06:00:00'}, 'arrival': {'iataCode': 'EWR', 'terminal': 'C', 'at': '2024-03-10T10:25:00'}, 'flightNumber': '1517', 'carrier': 'UNITED AIRLINES'}, {'departure': {'iataCode': 'EWR', 'termi

{'input': "Please draft a concise email from Santiago to Paul, Santiago's travel agent, asking him to book the earliest flight from DFW to DCA on March 10, 2024. Include all flight details in the email.",
 'output': 'We have found several flight options from Dallas Fort Worth (DFW) to Washington D.C. (DCA) on March 10, 2024. The earliest flight is with United Airlines, departing from DFW at 06:00 and arriving at DCA at 13:19 with flight numbers 1517 and 4431. The total price is 303.31 EURO.'}